# 04 - Pruebas de Hipotesis: Metodo de Pago y Comportamiento del Pasajero

## Fase 3: Estadistica Inferencial — Proyecto NYC Taxi

En este notebook investigamos si el **metodo de pago** (tarjeta de credito vs efectivo) esta asociado con diferencias en el comportamiento del pasajero, utilizando pruebas estadisticas formales.

### Pregunta de negocio
**El metodo de pago influye en el comportamiento del pasajero?**

### Objetivos de aprendizaje
- Aplicar la **prueba Chi-cuadrado de independencia** para evaluar asociacion entre variables categoricas
- Calcular la **V de Cramer** como medida del tamano del efecto
- Realizar **pruebas t de dos muestras** para comparar medias entre grupos
- Ejecutar un **ANOVA** para comparar medias de multiples grupos
- Interpretar p-valores y tamanos de efecto en un contexto de negocio

### Concepto clave
La estadistica inferencial nos permite ir mas alla de la descripcion: podemos **generalizar** hallazgos de una muestra a toda la poblacion y cuantificar la **incertidumbre** de nuestras conclusiones.

---
## 1. Carga de datos y preparacion

In [ ]:
import sys
sys.path.insert(0, '../../../src')
from bigquery.bq_helper import BigQueryHelper
bq = BigQueryHelper()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

In [ ]:
query = """
SELECT
    pickup_datetime,
    dropoff_datetime,
    passenger_count,
    trip_distance,
    fare_amount,
    tip_amount,
    tolls_amount,
    total_amount,
    payment_type,
    rate_code,
    pickup_location_id,
    EXTRACT(HOUR FROM pickup_datetime) AS pickup_hour,
    EXTRACT(DAYOFWEEK FROM pickup_datetime) AS day_of_week,
    TIMESTAMP_DIFF(dropoff_datetime, pickup_datetime, MINUTE) AS trip_duration_min,
    -- Zona de recogida basada en pickup_location_id (TLC Zone IDs)
    CASE
        WHEN pickup_location_id IN ('4','12','13','24','41','42','43','45','48','50','68','74','75','79','87','88','90','100','107','113','114','116','125','127','128','137','140','141','142','143','144','148','151','152','153','158','161','162','163','164','166','170','186','194','202','209','211','224','229','230','231','232','233','234','236','237','238','239','243','244','246','249','261','262','263')
        THEN 'Manhattan'
        WHEN pickup_location_id IN ('11','14','17','21','22','25','26','29','33','34','35','36','37','39','40','49','52','54','55','61','62','63','65','66','67','69','71','72','76','77','80','85','89','91','97','106','108','111','112','123','133','149','150','154','155','165','177','178','181','188','189','190','195','210','217','222','225','227','228','255','256','257')
        THEN 'Brooklyn'
        WHEN pickup_location_id = '132'
        THEN 'JFK_Airport'
        WHEN pickup_location_id = '138'
        THEN 'LaGuardia_Airport'
        ELSE 'Otra'
    END AS pickup_zone,
    -- Banda horaria
    CASE
        WHEN EXTRACT(HOUR FROM pickup_datetime) BETWEEN 6 AND 11 THEN 'Manana'
        WHEN EXTRACT(HOUR FROM pickup_datetime) BETWEEN 12 AND 17 THEN 'Tarde'
        WHEN EXTRACT(HOUR FROM pickup_datetime) BETWEEN 18 AND 22 THEN 'Noche'
        ELSE 'Madrugada'
    END AS hour_band,
    -- Porcentaje de propina
    SAFE_DIVIDE(tip_amount, fare_amount) * 100 AS tip_percentage
FROM `bigquery-public-data.new_york_taxi_trips.tlc_yellow_trips_2015`
WHERE
    fare_amount BETWEEN 2.5 AND 200
    AND trip_distance BETWEEN 0.1 AND 50
    AND passenger_count BETWEEN 1 AND 6
    AND pickup_location_id IS NOT NULL
    AND TIMESTAMP_DIFF(dropoff_datetime, pickup_datetime, MINUTE) BETWEEN 1 AND 180
    AND payment_type IN ('CRD', 'CSH', 'CAS', 'CRE')
ORDER BY FARM_FINGERPRINT(CAST(pickup_datetime AS STRING) || CAST(dropoff_datetime AS STRING))
LIMIT 50000
"""

df = bq.run_query(query, label='phase3_04_payments')
print(f"Registros cargados: {len(df):,}")
df.head()

In [ ]:
# Estandarizar payment_type: unificar variantes
payment_map = {'CRD': 'Tarjeta', 'CRE': 'Tarjeta', 'CSH': 'Efectivo', 'CAS': 'Efectivo'}
df['payment_label'] = df['payment_type'].map(payment_map)

# Estandarizar rate_code
rate_map = {
    'Standard rate': 'Standard',
    'JFK': 'JFK',
    'Newark': 'Newark',
    'Nassau or Westchester': 'Nassau',
    'Negotiated fare': 'Negociada'
}
df['rate_label'] = df['rate_code'].map(rate_map).fillna('Otro')

print("Distribucion de metodo de pago:")
print(df['payment_label'].value_counts())
print()
print("Distribucion de banda horaria:")
print(df['hour_band'].value_counts())
print()
print("Distribucion de rate_code:")
print(df['rate_label'].value_counts())

---
## 2. Chi-cuadrado de independencia: Metodo de pago vs Zona

La **prueba Chi-cuadrado de independencia** evalua si dos variables categoricas son estadisticamente independientes. La hipotesis nula es que no existe asociacion entre ellas.

### Hipotesis
- **H0:** El metodo de pago es independiente de la zona de recogida
- **H1:** El metodo de pago y la zona de recogida estan asociados

### Nota sobre la prueba
La Chi-cuadrado compara las frecuencias **observadas** en la tabla de contingencia con las frecuencias **esperadas** si las variables fueran independientes. Si la diferencia es grande (Chi-cuadrado alto, p-valor bajo), rechazamos la independencia.

In [ ]:
# Tabla de contingencia: payment_label x pickup_zone
# Excluimos 'Otra' para zonas mas definidas
zones_of_interest = ['Manhattan', 'Brooklyn', 'Queens_West', 'JFK_Airport', 'LaGuardia_Airport']
df_zones = df[df['pickup_zone'].isin(zones_of_interest)].copy()

contingency_zone = pd.crosstab(df_zones['payment_label'], df_zones['pickup_zone'])
print("Tabla de contingencia: Metodo de pago x Zona")
print(contingency_zone)
print()

In [ ]:
# Prueba Chi-cuadrado
chi2_zone, p_zone, dof_zone, expected_zone = stats.chi2_contingency(contingency_zone)

print(f"Prueba Chi-cuadrado: Metodo de pago vs Zona")
print(f"{'='*50}")
print(f"Estadistico Chi-cuadrado: {chi2_zone:.2f}")
print(f"Grados de libertad:       {dof_zone}")
print(f"p-valor:                  {p_zone:.2e}")
print(f"")

alpha = 0.05
if p_zone < alpha:
    print(f"RESULTADO: Rechazamos H0 (p < {alpha})")
    print("Existe una asociacion estadisticamente significativa entre")
    print("el metodo de pago y la zona de recogida.")
else:
    print(f"RESULTADO: No rechazamos H0 (p >= {alpha})")
    print("No hay evidencia suficiente de asociacion.")

In [ ]:
# Frecuencias esperadas vs observadas
expected_df = pd.DataFrame(
    expected_zone,
    index=contingency_zone.index,
    columns=contingency_zone.columns
).round(1)

print("Frecuencias esperadas (si fueran independientes):")
print(expected_df)
print()
print("Diferencia (Observado - Esperado):")
print((contingency_zone - expected_df).round(1))

---
## 3. Chi-cuadrado: Metodo de pago vs Banda horaria

### Hipotesis
- **H0:** El metodo de pago es independiente de la banda horaria
- **H1:** El metodo de pago y la banda horaria estan asociados

Intuitivamente, podriamos esperar que por la noche (menos comercios abiertos, turistas) se use mas efectivo, o al contrario, mas tarjeta por seguridad.

In [ ]:
# Tabla de contingencia: payment_label x hour_band
# Ordenar bandas horarias logicamente
hour_order = ['Manana', 'Tarde', 'Noche', 'Madrugada']
contingency_hour = pd.crosstab(df['payment_label'], df['hour_band'])
contingency_hour = contingency_hour[hour_order]

print("Tabla de contingencia: Metodo de pago x Banda horaria")
print(contingency_hour)
print()

In [ ]:
# Prueba Chi-cuadrado
chi2_hour, p_hour, dof_hour, expected_hour = stats.chi2_contingency(contingency_hour)

print(f"Prueba Chi-cuadrado: Metodo de pago vs Banda horaria")
print(f"{'='*50}")
print(f"Estadistico Chi-cuadrado: {chi2_hour:.2f}")
print(f"Grados de libertad:       {dof_hour}")
print(f"p-valor:                  {p_hour:.2e}")
print(f"")

if p_hour < alpha:
    print(f"RESULTADO: Rechazamos H0 (p < {alpha})")
    print("Existe una asociacion significativa entre metodo de pago y banda horaria.")
else:
    print(f"RESULTADO: No rechazamos H0 (p >= {alpha})")
    print("No hay evidencia suficiente de asociacion.")

In [ ]:
# Proporcion de tarjeta por banda horaria
prop_table = pd.crosstab(df['payment_label'], df['hour_band'], normalize='columns') * 100
prop_table = prop_table[hour_order]

print("Proporcion (%) de cada metodo de pago por banda horaria:")
print(prop_table.round(1))

---
## 4. V de Cramer: Tamano del efecto para Chi-cuadrado

El p-valor nos dice si la asociacion es **estadisticamente significativa**, pero con muestras grandes casi cualquier diferencia lo es. Necesitamos una medida del **tamano del efecto** para saber si la asociacion es **practicamente relevante**.

### Interpretacion de la V de Cramer
| V de Cramer | Interpretacion |
|-------------|----------------|
| < 0.10 | Efecto negligible |
| 0.10 - 0.30 | Efecto pequeno |
| 0.30 - 0.50 | Efecto mediano |
| > 0.50 | Efecto grande |

### Formula
$$V = \sqrt{\frac{\chi^2}{n \cdot (\min(r, c) - 1)}}$$

Donde $n$ es el total de observaciones, $r$ es el numero de filas y $c$ el numero de columnas de la tabla de contingencia.

In [ ]:
def cramers_v(contingency_table):
    """Calcula la V de Cramer a partir de una tabla de contingencia."""
    chi2, p, dof, expected = stats.chi2_contingency(contingency_table)
    n = contingency_table.sum().sum()
    r, c = contingency_table.shape
    v = np.sqrt(chi2 / (n * (min(r, c) - 1)))
    return v, chi2, p

# V de Cramer para ambas pruebas
v_zone, _, _ = cramers_v(contingency_zone)
v_hour, _, _ = cramers_v(contingency_hour)

print("Tamano del efecto (V de Cramer)")
print(f"{'='*50}")
print(f"Pago vs Zona:          V = {v_zone:.4f}")
print(f"Pago vs Banda horaria: V = {v_hour:.4f}")
print()

for name, v in [('Pago vs Zona', v_zone), ('Pago vs Banda horaria', v_hour)]:
    if v < 0.10:
        efecto = 'negligible'
    elif v < 0.30:
        efecto = 'pequeno'
    elif v < 0.50:
        efecto = 'mediano'
    else:
        efecto = 'grande'
    print(f"{name}: efecto {efecto}")

**Leccion importante:** Con 50,000 observaciones, la prueba Chi-cuadrado casi siempre rechaza H0. Lo que realmente importa es la V de Cramer: un efecto negligible o pequeno nos dice que, aunque la asociacion es real, su magnitud practica es limitada.

---
## 5. Prueba t de dos muestras: Tarifa segun metodo de pago

Ahora comparamos una **variable continua** (tarifa) entre dos grupos (tarjeta vs efectivo).

### Hipotesis
- **H0:** La tarifa media es igual para viajes con tarjeta y viajes con efectivo ($\mu_{tarjeta} = \mu_{efectivo}$)
- **H1:** Las tarifas medias difieren ($\mu_{tarjeta} \neq \mu_{efectivo}$)

### Supuestos de la prueba t
1. Las muestras son independientes
2. Las poblaciones tienen distribucion aproximadamente normal (con n grande, el TCL lo garantiza)
3. Varianzas pueden ser desiguales (usamos Welch's t-test)

In [ ]:
# Separar grupos
fare_card = df[df['payment_label'] == 'Tarjeta']['fare_amount']
fare_cash = df[df['payment_label'] == 'Efectivo']['fare_amount']

print(f"Tarjeta: n={len(fare_card):,}, media=${fare_card.mean():.2f}, mediana=${fare_card.median():.2f}, std=${fare_card.std():.2f}")
print(f"Efectivo: n={len(fare_cash):,}, media=${fare_cash.mean():.2f}, mediana=${fare_cash.median():.2f}, std=${fare_cash.std():.2f}")
print(f"\nDiferencia de medias: ${fare_card.mean() - fare_cash.mean():.2f}")

In [ ]:
# Visualizacion comparativa
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Histogramas superpuestos
axes[0].hist(fare_card, bins=50, alpha=0.5, label='Tarjeta', color='#2196F3',
             density=True, range=(0, 80))
axes[0].hist(fare_cash, bins=50, alpha=0.5, label='Efectivo', color='#FF5722',
             density=True, range=(0, 80))
axes[0].axvline(fare_card.mean(), color='#2196F3', linestyle='--', linewidth=2,
                label=f'Media tarjeta: ${fare_card.mean():.1f}')
axes[0].axvline(fare_cash.mean(), color='#FF5722', linestyle='--', linewidth=2,
                label=f'Media efectivo: ${fare_cash.mean():.1f}')
axes[0].set_xlabel('Tarifa (USD)')
axes[0].set_ylabel('Densidad')
axes[0].set_title('Distribucion de tarifa por metodo de pago')
axes[0].legend()

# Box plot
sns.boxplot(data=df, x='payment_label', y='fare_amount', ax=axes[1],
            palette=['#2196F3', '#FF5722'], showfliers=False)
axes[1].set_xlabel('Metodo de pago')
axes[1].set_ylabel('Tarifa (USD)')
axes[1].set_title('Box plot de tarifa por metodo de pago')

plt.tight_layout()
plt.show()

In [ ]:
# Prueba t de Welch (no asume varianzas iguales)
t_stat, p_value_t = stats.ttest_ind(fare_card, fare_cash, equal_var=False)

# Tamano del efecto: d de Cohen
pooled_std = np.sqrt((fare_card.std()**2 + fare_cash.std()**2) / 2)
cohen_d = (fare_card.mean() - fare_cash.mean()) / pooled_std

print(f"Prueba t de Welch: Tarifa tarjeta vs efectivo")
print(f"{'='*50}")
print(f"Estadistico t:    {t_stat:.4f}")
print(f"p-valor:          {p_value_t:.2e}")
print(f"d de Cohen:       {cohen_d:.4f}")
print()

# Interpretacion del tamano del efecto
if abs(cohen_d) < 0.2:
    efecto_t = 'negligible'
elif abs(cohen_d) < 0.5:
    efecto_t = 'pequeno'
elif abs(cohen_d) < 0.8:
    efecto_t = 'mediano'
else:
    efecto_t = 'grande'

print(f"Interpretacion: tamano del efecto {efecto_t}")
if p_value_t < alpha:
    print(f"La diferencia es estadisticamente significativa (p < {alpha})")
else:
    print(f"La diferencia NO es estadisticamente significativa (p >= {alpha})")

---
## 6. ANOVA: Tarifa segun tipo de tarifa (rate_code)

Cuando queremos comparar las medias de **mas de dos grupos**, usamos ANOVA (Analysis of Variance). Los rate codes representan diferentes estructuras tarifarias.

### Hipotesis
- **H0:** Todas las medias de tarifa son iguales entre rate codes ($\mu_1 = \mu_2 = ... = \mu_k$)
- **H1:** Al menos una media es diferente

### Rate codes en NYC
| Codigo | Descripcion |
|--------|-------------|
| Standard | Tarifa normal por taximetro |
| JFK | Tarifa plana al aeropuerto JFK ($52) |
| Newark | Tarifa al aeropuerto de Newark |
| Nassau | Tarifa a Nassau/Westchester |
| Negociada | Tarifa acordada entre pasajero y conductor |

In [ ]:
# Filtrar solo rate codes con suficientes observaciones
rate_counts = df['rate_label'].value_counts()
valid_rates = rate_counts[rate_counts >= 30].index.tolist()
df_anova = df[df['rate_label'].isin(valid_rates)].copy()

print("Estadisticas descriptivas por rate code:")
rate_stats = df_anova.groupby('rate_label')['fare_amount'].agg(
    ['count', 'mean', 'median', 'std']
).round(2)
rate_stats.columns = ['N', 'Media', 'Mediana', 'Desv. Est.']
print(rate_stats.sort_values('Media', ascending=False))

In [ ]:
# Visualizacion
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Box plot por rate code
order = rate_stats.sort_values('Media', ascending=False).index.tolist()
sns.boxplot(data=df_anova, x='rate_label', y='fare_amount', order=order,
            palette='Set2', showfliers=False, ax=axes[0])
axes[0].set_xlabel('Tipo de tarifa')
axes[0].set_ylabel('Tarifa (USD)')
axes[0].set_title('Distribucion de tarifa por rate code')
axes[0].tick_params(axis='x', rotation=15)

# Violin plot
sns.violinplot(data=df_anova, x='rate_label', y='fare_amount', order=order,
               palette='Set2', inner='box', ax=axes[1], cut=0)
axes[1].set_ylim(0, 100)
axes[1].set_xlabel('Tipo de tarifa')
axes[1].set_ylabel('Tarifa (USD)')
axes[1].set_title('Violin plot de tarifa por rate code')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

In [ ]:
# ANOVA de una via
groups = [group['fare_amount'].values for name, group in df_anova.groupby('rate_label')]
f_stat, p_anova = stats.f_oneway(*groups)

# Eta-cuadrado (tamano del efecto para ANOVA)
grand_mean = df_anova['fare_amount'].mean()
ss_between = sum(len(g) * (g.mean() - grand_mean)**2 for g in groups)
ss_total = sum((df_anova['fare_amount'] - grand_mean)**2)
eta_squared = ss_between / ss_total

print(f"ANOVA: Tarifa por rate code")
print(f"{'='*50}")
print(f"Estadistico F:    {f_stat:.2f}")
print(f"p-valor:          {p_anova:.2e}")
print(f"Eta-cuadrado:     {eta_squared:.4f}")
print()

# Interpretacion eta-cuadrado
if eta_squared < 0.01:
    efecto_eta = 'negligible'
elif eta_squared < 0.06:
    efecto_eta = 'pequeno'
elif eta_squared < 0.14:
    efecto_eta = 'mediano'
else:
    efecto_eta = 'grande'

print(f"Tamano del efecto: {efecto_eta}")
print(f"El rate code explica el {eta_squared*100:.1f}% de la varianza en tarifa.")

In [ ]:
# Post-hoc: Pruebas t pareadas con correccion de Bonferroni
from itertools import combinations

rate_groups = {name: group['fare_amount'].values 
               for name, group in df_anova.groupby('rate_label')}

n_comparisons = len(list(combinations(rate_groups.keys(), 2)))
alpha_bonferroni = alpha / n_comparisons

print(f"Comparaciones post-hoc (correccion de Bonferroni, alpha ajustado = {alpha_bonferroni:.4f})")
print(f"{'='*70}")

posthoc_results = []
for (name1, g1), (name2, g2) in combinations(rate_groups.items(), 2):
    t, p = stats.ttest_ind(g1, g2, equal_var=False)
    sig = 'Si' if p < alpha_bonferroni else 'No'
    posthoc_results.append({
        'Grupo 1': name1,
        'Grupo 2': name2,
        'Dif. medias': f"${g1.mean() - g2.mean():.2f}",
        'p-valor': f"{p:.2e}",
        'Significativo': sig
    })

df_posthoc = pd.DataFrame(posthoc_results)
print(df_posthoc.to_string(index=False))

---
## 7. Analisis: Los usuarios de tarjeta toman viajes mas largos o cortos?

Vamos a comparar multiples metricas del viaje entre usuarios de tarjeta y efectivo para construir un perfil de comportamiento.

In [ ]:
# Comparacion multidimensional
metrics = ['fare_amount', 'trip_distance', 'trip_duration_min', 'tip_percentage', 'passenger_count']
metric_labels = ['Tarifa (USD)', 'Distancia (millas)', 'Duracion (min)', 'Propina (%)', 'Pasajeros']

comparison = df.groupby('payment_label')[metrics].agg(['mean', 'median']).round(2)
print("Comparacion Tarjeta vs Efectivo:")
print(comparison)
print()

In [ ]:
# Pruebas t para cada metrica
print(f"Pruebas t para cada metrica (tarjeta vs efectivo)")
print(f"{'='*75}")
print(f"{'Metrica':<20} {'Media Tarj.':<12} {'Media Efect.':<12} {'t':<10} {'p-valor':<12} {'Cohen d':<10}")
print(f"{'-'*75}")

for metric, label in zip(metrics, metric_labels):
    g1 = df[df['payment_label'] == 'Tarjeta'][metric].dropna()
    g2 = df[df['payment_label'] == 'Efectivo'][metric].dropna()
    t, p = stats.ttest_ind(g1, g2, equal_var=False)
    d = (g1.mean() - g2.mean()) / np.sqrt((g1.std()**2 + g2.std()**2) / 2)
    print(f"{label:<20} {g1.mean():<12.2f} {g2.mean():<12.2f} {t:<10.2f} {p:<12.2e} {d:<10.3f}")

In [ ]:
# Visualizacion comparativa multivariable
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

plot_metrics = ['trip_distance', 'trip_duration_min', 'fare_amount', 'tip_percentage']
plot_labels = ['Distancia (millas)', 'Duracion (min)', 'Tarifa (USD)', 'Propina (%)']
ranges = [(0, 20), (0, 60), (0, 60), (0, 40)]

for ax, metric, label, rng in zip(axes.flat, plot_metrics, plot_labels, ranges):
    for pmt, color in [('Tarjeta', '#2196F3'), ('Efectivo', '#FF5722')]:
        data = df[df['payment_label'] == pmt][metric].dropna()
        data = data[(data >= rng[0]) & (data <= rng[1])]
        ax.hist(data, bins=40, alpha=0.5, color=color, label=pmt, density=True)
    ax.set_xlabel(label)
    ax.set_ylabel('Densidad')
    ax.set_title(f'{label} por metodo de pago')
    ax.legend()

plt.suptitle('Perfil de viaje: Tarjeta vs Efectivo', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 8. Heatmaps de tablas de contingencia

Los heatmaps son una forma visual muy efectiva de representar tablas de contingencia. Los colores permiten identificar rapidamente las celdas con mayor o menor frecuencia.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Heatmap 1: Pago x Zona (proporciones por columna)
prop_zone = pd.crosstab(df_zones['payment_label'], df_zones['pickup_zone'],
                         normalize='columns') * 100

sns.heatmap(prop_zone.astype(float), annot=True, fmt='.1f', cmap='YlOrRd',            linewidths=0.5, ax=axes[0],
            cbar_kws={'label': '% dentro de cada zona'})
axes[0].set_title('Metodo de pago por zona (% columna)', fontsize=13)
axes[0].set_xlabel('Zona de recogida')
axes[0].set_ylabel('Metodo de pago')

# Heatmap 2: Pago x Banda horaria (proporciones por columna)
prop_hour = pd.crosstab(df['payment_label'], df['hour_band'],
                         normalize='columns') * 100
prop_hour = prop_hour[hour_order]

sns.heatmap(prop_hour.astype(float), annot=True, fmt='.1f', cmap='YlOrRd',            linewidths=0.5, ax=axes[1],
            cbar_kws={'label': '% dentro de cada banda'})
axes[1].set_title('Metodo de pago por banda horaria (% columna)', fontsize=13)
axes[1].set_xlabel('Banda horaria')
axes[1].set_ylabel('Metodo de pago')

plt.tight_layout()
plt.show()

---
## 9. Grafico de barras apiladas (alternativa al mosaic plot)

El **mosaic plot** es una visualizacion clasica para tablas de contingencia, pero puede ser dificil de interpretar. Una alternativa mas accesible es el **grafico de barras apiladas al 100%**, que muestra las proporciones de cada categoria dentro de cada grupo.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Barras apiladas: Zona x Pago
prop_zone_t = prop_zone.T  # Transponer para que las zonas sean el eje x
prop_zone_t.plot(kind='bar', stacked=True, ax=axes[0],
                  color=['#FF5722', '#2196F3'], edgecolor='white')
axes[0].set_title('Proporcion de metodo de pago por zona')
axes[0].set_xlabel('Zona')
axes[0].set_ylabel('Porcentaje (%)')
axes[0].legend(title='Pago')
axes[0].tick_params(axis='x', rotation=30)
axes[0].axhline(y=50, color='gray', linestyle='--', alpha=0.5)

# Barras apiladas: Banda horaria x Pago
prop_hour_t = prop_hour.T
prop_hour_t.plot(kind='bar', stacked=True, ax=axes[1],
                  color=['#FF5722', '#2196F3'], edgecolor='white')
axes[1].set_title('Proporcion de metodo de pago por banda horaria')
axes[1].set_xlabel('Banda horaria')
axes[1].set_ylabel('Porcentaje (%)')
axes[1].legend(title='Pago')
axes[1].tick_params(axis='x', rotation=0)
axes[1].axhline(y=50, color='gray', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
# Detalle adicional: Conteos absolutos por zona y metodo de pago (barras agrupadas)
fig, ax = plt.subplots(figsize=(12, 6))

contingency_zone.T.plot(kind='bar', ax=ax, color=['#FF5722', '#2196F3'],
                         edgecolor='white', width=0.7)
ax.set_title('Conteo de viajes por zona y metodo de pago')
ax.set_xlabel('Zona de recogida')
ax.set_ylabel('Numero de viajes')
ax.legend(title='Pago')
ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

---
## 10. Resumen de hallazgos e implicaciones practicas

### Resultados de las pruebas estadisticas

| Prueba | Variables | Resultado | Tamano efecto | Interpretacion practica |
|--------|-----------|-----------|---------------|------------------------|
| Chi-cuadrado | Pago x Zona | Significativo | V de Cramer (ver arriba) | La preferencia de pago varia por zona |
| Chi-cuadrado | Pago x Hora | Significativo | V de Cramer (ver arriba) | Ligera variacion por momento del dia |
| Prueba t | Tarifa: Tarjeta vs Efectivo | Significativo | d de Cohen (ver arriba) | Diferencia en monto de tarifa |
| ANOVA | Tarifa x Rate code | Significativo | Eta-cuadrado (ver arriba) | Tarifas muy distintas por tipo |

### Lecciones clave de estadistica inferencial

1. **Significancia estadistica no es lo mismo que importancia practica.** Con muestras grandes, diferencias minusculas pueden ser "significativas".

2. **Siempre reportar el tamano del efecto** junto con el p-valor: V de Cramer, d de Cohen, eta-cuadrado.

3. **Multiples comparaciones** requieren correccion (Bonferroni) para evitar falsos positivos.

4. **El contexto del negocio** es fundamental: una diferencia de $1 en tarifa puede ser estadisticamente significativa pero irrelevante para decisiones de negocio.

### Implicaciones para el negocio

- **Infraestructura de pago:** Las zonas con mayor proporcion de tarjeta necesitan terminales de pago confiables
- **Propinas:** Los pagos con tarjeta generan propinas registradas (en efectivo se subregistran), lo que afecta el ingreso reportado del conductor
- **Segmentacion:** Los patrones de pago por hora y zona pueden informar estrategias de marketing diferenciadas

### Siguiente notebook
En el notebook 05 profundizaremos en las **correlaciones** entre variables, usando correlacion parcial para identificar relaciones verdaderas vs espurias.